Retrieve [research topics from OpenAlex](https://openalex.org/works?page=1&filter=cites%3Aw2015197254) for all works that cite the original pymatgen paper.

Then use LLM to organize topics into a mindmap.

Note:
- You would need to get `OPENAI_API_KEY` to use OpenAI API, see https://openai.com/api/.

In [ ]:
!uv pip install requests openai

NUM_OF_MAIN_TOPICS: int = 5
NUM_OF_SUB_TOPICS: int = 4

Using Python 3.12.11 environment at: /Users/yang/developer/pymatgen-2-paper/.venv
Audited 2 packages in 10ms


In [ ]:
# Step 1: Get topics from OpenAlex for 1st pymatgen paper

from collections import Counter

import requests

params = {"filter": "cites:w2015197254", "group_by": "primary_topic.id", "per_page": 50}
response = requests.get(
    "https://api.openalex.org/works", params=params, timeout=10
).json()

topic_counter = Counter(
    {
        group.get("key_display_name", "None"): group["count"]
        for group in response.get("group_by")
        if group["count"] >= 10  # Count for top 50 topics is around 10
    }
)

print("Topics from OpenAlex:")
for topic, count in topic_counter.items():
    print(f"  {topic:<60}{count}")

Topics from OpenAlex:
  Machine Learning in Materials Science                       1153
  Advancements in Battery Materials                           323
  Advanced Battery Materials and Technologies                 322
  Perovskite Materials and Applications                       107
  Metal-Organic Frameworks: Synthesis and Applications        74
  Electrocatalysts for Energy Conversion                      72
  Electronic and Structural Properties of Oxides              67
  2D Materials and Applications                               66
  Advanced Thermoelectric Materials and Devices               55
  MXene and MAX Phase Materials                               49
  Synthesis and Properties of Inorganic Cluster Compounds     41
  X-ray Diffraction in Crystallography                        37
  Chalcogenide Semiconductor Thin Films                       35
  Advancements in Solid Oxide Fuel Cells                      31
  High Entropy Alloys Studies                                 3

In [ ]:
# Step 2: Use LLM to group topics

import re

from openai import OpenAI

client = OpenAI()  # https://platform.openai.com/docs/overview

topic_list = "\n".join(f"- {topic} ({count})" for topic, count in topic_counter.items())

prompt = f"""
The following research topics are from papers that cite pymatgen, with each number indicating the count of citing works in that area.

Please organize them into {NUM_OF_MAIN_TOPICS} thematic branches that represent broader areas of materials research (e.g., batteries, machine learning, catalysis, ...).

Return output in a plain text tree structure (with indentation) without any header or metadata, e.g.:
    Topic_0:
        subtopic_0 (count_0)
        subtopic_1 (count_1)

    Topic_1:
        subtopic_0 (count_0)

Topics to work on today:
    {topic_list}
"""

chat_response = client.responses.create(
    model="gpt-4.1",
    input=prompt,
)

# print(chat_response.output_text)


def parse_mindmap(text: str) -> dict[str, list[tuple[str, int]]]:
    """
    Parse LLM response to machine-readable dict for plotter.
    """
    mindmap: dict[str, list[tuple[str, int]]] = {}
    current_topic = None

    for line in text.strip().splitlines():
        line = line.rstrip()

        if not line.strip():
            continue

        if not line.startswith(" "):  # Top-level topic
            current_topic = line.strip().removesuffix(":")
            mindmap[current_topic] = []

        elif current_topic:
            subtopic_line = line.strip()

            # Extract the (name, count)
            if match := re.match(r"^(.*)\s+\((\d+)\)$", subtopic_line):
                name = match[1].strip()
                count = int(match[2])
                mindmap[current_topic].append((name, count))
            else:
                raise ValueError(f"Cannot extract info from {subtopic_line=}")

    return mindmap


mindmap_dict: dict[str, list[tuple[str, int]]] = parse_mindmap(
    chat_response.output_text
)

# Sort main topics by the sum of subtopic counts
sorted_main_topics = sorted(
    mindmap_dict.items(),
    key=lambda item: sum(count for _, count in item[1]),
    reverse=True,
)

# Limit max number of main/sub topics
mindmap_dict = {
    main_topic: sorted(subtopics, key=lambda x: x[1], reverse=True)[:NUM_OF_SUB_TOPICS]
    for main_topic, subtopics in sorted_main_topics[:NUM_OF_MAIN_TOPICS]
}

for main_topic, subtopics in mindmap_dict.items():
    print(f"{main_topic}:")
    for name, count in subtopics:
        print(f"    - {name} ({count})")

Machine Learning and Computational Methods:
    - Machine Learning in Materials Science (1153)
    - Scientific Computing and Data Management (14)
    - Computational Drug Discovery Methods (14)
Energy Materials and Devices:
    - Advancements in Battery Materials (323)
    - Advanced Battery Materials and Technologies (322)
    - Advanced Thermoelectric Materials and Devices (55)
    - Advancements in Solid Oxide Fuel Cells (31)
Advanced Materials and Nanomaterials:
    - Perovskite Materials and Applications (107)
    - Metal-Organic Frameworks: Synthesis and Applications (74)
    - 2D Materials and Applications (66)
    - MXene and MAX Phase Materials (49)
Semiconductors, Oxides, and Electronic Materials:
    - Electronic and Structural Properties of Oxides (67)
    - Chalcogenide Semiconductor Thin Films (35)
    - Magnetic and transport properties of perovskites and related materials (24)
    - Semiconductor materials and devices (18)
Catalysis and Chemical Processes:
    - Electr

In [ ]:
# Step 3: Generate mindmap plot with Latex
# Reference: https://github.com/janosh/diagrams/blob/main/assets/physics-mindmap/physics-mindmap.tex

import os
import subprocess

mindmap_filename: str = "pymatgen_mindmap.tex"


def write_tikz_mindmap(
    mindmap: dict[str, list[tuple[str, int]]],
    filename: str,
) -> None:
    r"""Note for tweaking angles in the mindmap:

    There're two `sibling angle` at the "root" level (`\begin{tikzpicture}`),
    they would control the angle between each sibling.

    Meanwhile in the node level there's `[clockwise from=0]`, which controls
    the "starting angle" for this node (could be children).

    Note: angle `0` corresponds to the positive x-axis direction.
    """
    # colors: https://tex.stackexchange.com/questions/653991/how-can-i-use-more-colors-in-tikz
    tikz_colors: list[str] = [
        "Gold",
        "Teal",
        "Red",
        "Purple",
        "DarkGreen",
        "RoyalBlue",
        "Orchid",
        "Brown",
        "CornflowerBlue",
        "SlateGray",
    ][: len(mindmap)]

    def escape_latex(text: str) -> str:
        return (
            text.replace("&", r"\&")
            .replace("-", r"\-")
            .replace("_", r"\_")
            .replace("%", r"\%")
            .replace("#", r"\#")
        )

    with open(filename, "w", encoding="utf-8") as f:
        f.write(
            r"""\documentclass[tikz, svgnames]{standalone}
\usetikzlibrary{mindmap}
\begin{document}
\begin{tikzpicture}[
    mindmap, every node/.style=concept, concept color=orange, text=white,
    level 1/.append style={level distance=6cm, sibling angle=72, font=\large},
    level 2/.append style={level distance=4cm, sibling angle=45}
  ]

  \node{\textbf{\huge{pymatgen}}} [clockwise from=0]
"""
        )

        # TODO: need programmatic way to determine the node/child angles
        children_angles: list[float] = [45, 10, -60, 200, 120]
        assert len(children_angles) == NUM_OF_MAIN_TOPICS

        for i, (main_topic, subtopics) in enumerate(mindmap.items()):
            color = tikz_colors[i % len(tikz_colors)]
            f.write(f"  child [concept color={color}, text=black] {{\n")
            f.write(
                f"      node {{{escape_latex(main_topic)}}} [clockwise from={children_angles[i]}] \n"
            )
            for name, _count in subtopics:
                f.write(f"      child {{ node {{{escape_latex(name)}}} }}\n")
            f.write("    }\n")

        f.write(r""";
\end{tikzpicture}
\end{document}
""")


write_tikz_mindmap(mindmap_dict, mindmap_filename)

try:
    os.path.splitext(mindmap_filename)[0] + ".pdf"
    result = subprocess.run(
        ["pdflatex", "-interaction=nonstopmode", mindmap_filename],
        check=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
    )
except subprocess.CalledProcessError as e:
    print("❌ LaTeX compilation failed:")
    print("stdout:")
    print(e.stdout + "\n")
    print("stderr:")
    print(e.stderr)

finally:
    for ext in (".aux", ".log"):
        try:
            os.remove(os.path.splitext(mindmap_filename)[0] + ext)
        except FileNotFoundError:
            pass